# Pack 01 — Sampling, Aliasing and Quantisation

**Module 05-dsp · Etherstack interview prep · all C++ exercises live in `code/`; this notebook is for the pictures and numbers.**

| | |
|---|---|
| Time | ~90 min (30 min reading/demos, 30 min pen-and-paper, 30 min C++) |
| You need | Python 3 + NumPy + Matplotlib for the demos; CMake + a C++17 compiler for `code/` |
| Signals | 100 % synthetic — no captured RF, nothing proprietary |

**How to use this notebook**
1. Run every code cell and *look* at each plot before reading the derivation under it.
2. Do the pen-and-paper exercises (Part A) without the answers.
3. Implement the C++ stubs (Part B) until `ctest` is green.
4. Rehearse Part C out loud — those are interview-style questions.

**Why Etherstack cares.** A land-mobile-radio (P25 / DMR / TETRA style) receiver is, at its front end, an ADC followed by decimation. Every later stage (timing recovery, demodulation, BER) assumes the samples faithfully represent the band of interest. Aliasing and quantisation noise are the two ways the ADC *breaks* that assumption, and "explain aliasing / what SNR does a 12-bit ADC give" is a classic first question.

## 1. Concrete first: two different tones, identical samples

Before any theory: sample a **7 kHz** cosine and a **1 kHz** cosine at **fs = 8 kHz** and compare the numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 8000.0
n = np.arange(12)
x7 = np.cos(2 * np.pi * 7000.0 * n / fs)
x1 = np.cos(2 * np.pi * 1000.0 * n / fs)
print("7 kHz samples:", np.round(x7, 4))
print("1 kHz samples:", np.round(x1, 4))
print("identical?    ", np.allclose(x7, x1))

# sine version: the sign flips (negative frequency!)
s7 = np.sin(2 * np.pi * 7000.0 * n / fs)
s1 = np.sin(2 * np.pi * 1000.0 * n / fs)
print("sin 7 kHz == -sin 1 kHz ?", np.allclose(s7, -s1))

t = np.linspace(0, 1.5e-3, 3000)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(t * 1e3, np.cos(2 * np.pi * 7000.0 * t), lw=1, label="7 kHz cosine (true)")
ax.plot(t * 1e3, np.cos(2 * np.pi * 1000.0 * t), lw=1, label="1 kHz cosine (alias)")
ax.plot(n / fs * 1e3, x7, "ko", label="samples @ 8 kS/s")
ax.set_xlabel("time [ms]"); ax.set_ylabel("amplitude"); ax.legend(loc="lower right")
ax.set_title("Both curves pass through every sample — the ADC cannot tell them apart")
plt.tight_layout(); plt.show()

### 1.1 Derivation: why 7 kHz looks like 1 kHz

Sampling instants are $t_n = n/f_s$. A cosine of frequency $f$ gives

$$x[n] = \cos(2\pi f n / f_s).$$

Add any integer multiple $k$ of $f_s$ to the frequency:

$$\cos\!\big(2\pi (f + k f_s) n / f_s\big) = \cos\!\big(2\pi f n/f_s + 2\pi k n\big) = \cos(2\pi f n / f_s),$$

because $kn$ is an integer so $2\pi k n$ is a whole number of turns. **Every frequency $f + k f_s$ produces exactly the same sample stream.** In the demo $7000 = -1000 + 8000$ and cosine is even, so it looks like 1 kHz; for sine, which is odd, the sign flips.

**Rule (derived, not memorised):** the apparent frequency is $f$ reduced modulo $f_s$ into $[-f_s/2, f_s/2]$; for a *real* tone only $|f|$ is observable, so it lands in $[0, f_s/2]$.

```
r = |f| mod fs
f_apparent = r            if r <= fs/2
           = fs - r       otherwise
```

That is exactly `alias_frequency()` in Exercise B1.

In [ ]:
def alias(f, fs):
    r = np.mod(np.abs(f), fs)
    return np.where(r > fs / 2, fs - r, r)

f = np.linspace(0, 3 * fs, 2000)
plt.figure(figsize=(8, 3.5))
plt.plot(f / 1e3, alias(f, fs) / 1e3)
plt.xlabel("true tone frequency [kHz]"); plt.ylabel("apparent frequency [kHz]")
plt.title("Folding: apparent vs true frequency (fs = 8 kHz)")
plt.grid(True); plt.tight_layout(); plt.show()

# Same thing seen through a DFT: four different true frequencies, one peak.
N = 64; nn = np.arange(N)
for ftrue in (1000, 7000, 9000, 15000):
    X = np.abs(np.fft.rfft(np.cos(2 * np.pi * ftrue * nn / fs)))
    print(f"true {ftrue:6d} Hz -> DFT peak at {X.argmax() * fs / N:7.1f} Hz")

### 1.2 Nyquist, stated carefully

A signal whose spectrum lies entirely inside $|f| < f_s/2$ is uniquely determined by its samples. The condition is on **bandwidth**, not on "the highest frequency we care about": energy *anywhere* (noise, an adjacent channel, a spur) above $f_s/2$ folds in on top of your wanted signal and **cannot be removed afterwards**. Hence the analogue **anti-alias filter** *before* the ADC.

**Edge case — exactly $f_s/2$.** $\sin(2\pi (f_s/2) n/f_s) = \sin(\pi n) = 0$ for all $n$. A sine at exactly Nyquist can vanish completely, a cosine survives as $(-1)^n$. So the strict inequality matters.

### 1.3 Misconceptions

| Wrong | Right |
|---|---|
| "Sampling at 2f means I keep a tone at f" | Need $f_s > 2f$ strictly; at equality phase decides if you see anything |
| "I only care about 0–6 kHz so fs = 12 kHz is fine" | Everything in $f_s/2 \pm$ guard folds in; you need a filter *and* transition-band margin |
| "Aliasing is a software bug I can fix later" | It is irreversible after sampling |
| "Nyquist means baseband only" | **Band-pass** signals can be sampled below 2·$f_{max}$ — see next section |

## 2. Band-pass (under-)sampling — used on purpose in IF-sampling receivers

If the wanted signal occupies only $[f_L, f_H]$ with bandwidth $B = f_H - f_L$, we can *deliberately* alias it down to a low frequency without overlap.

**Derivation.** Sampling at $f_s$ replicates the positive-frequency band at $[f_L + k f_s, f_H + k f_s]$ and its negative-frequency mirror at $[-f_H + k f_s, -f_L + k f_s]$. Choose integer $k$ so the band sits between two multiples of $f_s/2$:

$$ (k-1)\tfrac{f_s}{2} \;\le\; f_L,\qquad f_H \;\le\; k\tfrac{f_s}{2} \;\;\Longleftrightarrow\;\; \frac{2f_H}{k} \le f_s \le \frac{2 f_L}{k-1}.$$

Then positive and negative images never overlap. The largest usable $k$ is $\lfloor f_H/B \rfloor$ and gives the lowest rate (never below $2B$).

**Worked example (synthetic):** band 21–26 MHz, $B=5$ MHz, $k=\lfloor 26/5 \rfloor = 5$:
$2\cdot26/5 = 10.4\ \text{MHz} \le f_s \le 2\cdot 21/4 = 10.5\ \text{MHz}$. So $f_s = 10.4$ MHz works — about 5× less than $2 f_H = 52$ MHz — **but** the window is only 100 kHz wide: the sample clock needs excellent accuracy, and the analogue band-pass filter must be sharp.

In [ ]:
def bandpass_ok(fl, fh, fs):
    """True if [fl,fh] folds without overlapping its own mirror."""
    for k in range(1, int(np.floor(fh / (fh - fl))) + 1):
        lo = 2 * fh / k
        hi = 2 * fl / (k - 1) if k > 1 else np.inf
        if lo <= fs <= hi:
            return True
    return False

for fs_try in (10.0, 10.4, 10.45, 10.5, 10.6, 12.0):
    print(f"fs = {fs_try:5.2f} MHz  ->  21-26 MHz band OK? {bandpass_ok(21, 26, fs_try)}")

## 3. Quantisation: from a continuous amplitude to $N$ bits

**Concrete first** — quantise a full-scale sine with a 3-bit mid-tread quantiser and look at the *error* $e[n] = q[n] - x[n]$.

In [ ]:
def quantise(x, bits, full_scale=1.0):
    delta = 2 * full_scale / 2 ** bits
    k = np.clip(np.floor(x / delta + 0.5), -2 ** (bits - 1), 2 ** (bits - 1) - 1)
    return k * delta

bits = 3
delta = 2.0 / 2 ** bits
A = 1.0 - delta                   # largest amplitude that never clips
n = np.arange(200)
x = A * np.sin(2 * np.pi * 0.0137 * n)
q = quantise(x, bits)
e = q - x
fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
ax[0].plot(n, x, label="x"); ax[0].step(n, q, where="mid", label="q"); ax[0].legend(); ax[0].set_title(f"{bits}-bit quantiser (Δ = {delta})")
ax[1].plot(n, e); ax[1].axhline(delta / 2, ls="--"); ax[1].axhline(-delta / 2, ls="--"); ax[1].set_title("error stays within ±Δ/2")
ax[2].hist(e, bins=20); ax[2].set_title("error histogram: roughly flat → uniform assumption")
plt.tight_layout(); plt.show()

### 3.1 Derivation: SNR ≈ 6.02 N + 1.76 dB

**Model.** $N$ bits give $2^N$ levels across $[-A, +A]$ so the step is $\Delta = 2A / 2^N$.

**Assumption (the "uniform-noise model").** The rounding error $e$ is uniformly distributed on $[-\Delta/2, +\Delta/2]$, uncorrelated with the signal, white. That is plausible when the signal is "busy" (crosses many levels) and does not clip.

1. Density $p(e) = 1/\Delta$ on $[-\Delta/2,\Delta/2]$.
2. Noise power
$$P_e = \int_{-\Delta/2}^{\Delta/2} e^2 \frac{1}{\Delta}\,de = \frac{1}{\Delta}\cdot\frac{2(\Delta/2)^3}{3} = \frac{\Delta^2}{12}.$$
3. Full-scale sine of amplitude $A$: $P_s = A^2/2$.
4. Substitute $\Delta^2 = 4A^2/2^{2N}$:
$$\text{SQNR} = \frac{A^2/2}{\Delta^2/12} = \frac{A^2/2}{4A^2/(12\cdot 2^{2N})} = \tfrac{3}{2}\, 2^{2N}.$$
5. In dB: $10\log_{10}(1.5) + 20N\log_{10}2 = \mathbf{1.76 + 6.02\,N}$ dB.

**Meaning:** each extra bit halves $\Delta$, quartering the noise power: **+6.02 dB per bit**. The 1.76 dB is a property of a *sine* (peak/RMS = √2); a different signal shape changes it.

In [ ]:
def sqnr_measured(bits, npts=1 << 16, f=0.123456789):
    delta = 2.0 / 2 ** bits
    A = 1.0 - delta
    x = A * np.sin(2 * np.pi * f * np.arange(npts))
    e = quantise(x, bits) - x
    return 10 * np.log10(np.mean(x ** 2) / np.mean(e ** 2))

bs = np.arange(2, 17)
meas = np.array([sqnr_measured(b) for b in bs])
theo = 6.02 * bs + 1.76
for b, m, t_ in zip(bs, meas, theo):
    print(f"N={b:2d}  measured {m:6.2f} dB   6.02N+1.76 = {t_:6.2f} dB   diff {m - t_:+.2f}")
plt.figure(figsize=(7, 4)); plt.plot(bs, theo, "-", label="6.02N+1.76"); plt.plot(bs, meas, "o", label="simulated")
plt.xlabel("bits N"); plt.ylabel("SQNR [dB]"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

### 3.2 Where the model breaks (read this before quoting "6.02N+1.76")

1. **Not full scale.** A signal at −12 dBFS loses 12 dB: $\text{SNR} = 6.02N+1.76 - \text{back-off}$.
2. **High peak-to-average ratio (PAPR).** Multi-carrier / OFDM / many-user composite signals need ~10 dB headroom so they do not clip: a 14-bit ADC then behaves like ~12.3 bits.
3. **Signal periodic with the sample clock.** The error is then also periodic → **spurs (harmonics), not white noise.** Demo below: tone at exactly $f_s/16$.
4. **Clipping** adds large non-uniform errors, ADC thermal noise, jitter and DNL/INL set the real **ENOB**, always less than nominal bits.

In [ ]:
def spectrum_db(e, nwin=1 << 14):
    w = np.hanning(nwin)
    E = np.abs(np.fft.rfft(e[:nwin] * w)) ** 2
    return 10 * np.log10(E + 1e-30)

bits = 8
delta = 2.0 / 2 ** bits
A = 1.0 - delta
n = np.arange(1 << 14)
rng = np.random.default_rng(1)
cases = {"incommensurate tone": 0.123456789, "tone at fs/16 (periodic)": 1 / 16}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (name, f) in zip(axs, cases.items()):
    x = A * np.sin(2 * np.pi * f * n)
    e = quantise(x, bits) - x
    S = spectrum_db(e)
    ax.plot(S, lw=0.6); ax.set_title(name)
# TPDF dither (two uniform variables of ±Δ/2) before quantising the periodic tone
x = A * np.sin(2 * np.pi * (1 / 16) * n)
d = (rng.uniform(-0.5, 0.5, n.size) + rng.uniform(-0.5, 0.5, n.size)) * delta
e_d = quantise(x + d, bits) - x
axs[2].plot(spectrum_db(e_d), lw=0.6); axs[2].set_title("fs/16 tone + TPDF dither")
for ax in axs: ax.set_xlabel("DFT bin")
axs[0].set_ylabel("error spectrum [dB]")
plt.tight_layout(); plt.show()

### 3.3 Oversampling buys SNR — and why decimation filters matter

The total noise power $\Delta^2/12$ is fixed, but it is spread uniformly over $[0, f_s/2]$. If the wanted signal only occupies bandwidth $B$ and we filter to $B$, the surviving noise is

$$P_{e,\text{in-band}} = \frac{\Delta^2}{12}\cdot\frac{B}{f_s/2}.$$

Doubling $f_s$ halves in-band noise → **+3 dB per doubling** ($10\log_{10}\text{OSR}$). Check it numerically: the fraction of error energy in the lowest $1/\text{OSR}$ of the spectrum should be $\approx 1/\text{OSR}$.

In [ ]:
bits = 8
delta = 2.0 / 2 ** bits
A = 1.0 - delta
N = 1 << 16
for osr in (1, 2, 4, 8):
    f0 = 0.123456789 / osr                    # tone stays at the same *analogue* frequency
    x = A * np.sin(2 * np.pi * f0 * np.arange(N))
    e = quantise(x, bits) - x
    E = np.abs(np.fft.rfft(e * np.hanning(N))) ** 2
    frac = E[: len(E) // osr].sum() / E.sum()
    print(f"OSR {osr}: in-band noise fraction {frac:.3f}  (1/OSR = {1/osr:.3f})  gain = {-10*np.log10(frac):.2f} dB")

## 4. Etherstack / radio context

* **Channel rates.** A 12.5 kHz LMR channel is narrow; the ADC runs at MHz rates and a digital down-converter + decimator (Pack 04, 09) reduces to a few samples per symbol. The decimation filter is the *digital* anti-alias filter for the next rate change — same aliasing rule, now applied in software.
* **Dynamic range.** A receiver must handle a weak wanted signal next to a strong adjacent-channel transmitter. ADC bits set the floor; AGC (Pack 07) keeps the wanted signal near full scale after analogue gain.
* **Fixed-point DSP** (Pack 09): the same $6.02N$ rule applies to internal Q15 arithmetic: 16-bit data → ≤ ~98 dB in theory, less with headroom.

## 5. Exercises

### Part A — pen and paper
1. **Alias table.** For $f_s = 8$ kHz give the apparent frequency of tones at 3 kHz, 5 kHz, 7 kHz, 13 kHz, 16.5 kHz, 4 kHz.
2. For $f_s = 48$ kHz a spur appears at 30 kHz at the input. Where does it land? What if the ADC rate is 44.1 kHz?
3. A 12-bit ADC (±1 V full-scale) is fed a sine of 0.5 V amplitude. Compute Δ, noise power, and SQNR in dB. Compare with 6.02N+1.76.
4. How many bits are needed for 90 dB SQNR on a full-scale sine? What if the signal has 12 dB PAPR-driven back-off from a sine?
5. Band 21–26 MHz: find the **range** of valid sampling frequencies for $k = 5$ and $k=4$. Is $k=4$ valid? 
6. Show that a sine at exactly $f_s/2$ with phase 0 samples to all zeros; what phase gives maximum amplitude?
7. A system oversamples by 16× then decimates. How much does the in-band quantisation SQNR improve (assume white error)? How many "equivalent bits" is that?
8. **Prove** that if the error were *not* uniform but uniform on $[-\Delta, 0]$ (truncation instead of rounding) the noise *power* changes — compute it. (Hint: it has non-zero mean.)

### Part B — C++ (in `code/`)
| # | Function | Teaches |
|---|---|---|
| B1 | `alias_frequency` | modulo-fold rule |
| B2 | `sample_sine` | sampling instants, phase |
| B3 | `quantise_uniform` | mid-tread rounding, clipping |
| B4 | `ideal_sqnr_db`, `measured_sqnr_db` | theory vs experiment |
| B5 | `bandpass_fs_min` | band-pass sampling |

Build: `cmake -S code -B build && cmake --build build && ctest --test-dir build --output-on-failure` (stubs fail on purpose). Reference: add `-DUSE_SOLUTIONS=ON`.

### Part C — interview questions (answer aloud in 60 s each)
1. What is aliasing and how do you prevent it in a receiver chain?
2. Derive 6.02N + 1.76 on a whiteboard.
3. Your 14-bit ADC data shows only ~72 dB SNR. List five reasons.
4. Why can band-pass sampling work, and what does it cost you?
5. A customer sees a phantom tone that moves *opposite* to the tuning knob. Explain. (Hint: which alias/inversion?)

## 6. VERIFY list for this pack

* The "ENOB < nominal bits" statement is general knowledge; datasheet numbers for any real ADC must be checked against the part's data sheet.
* The 6.02N+1.76 constant is derived above (exact: 6.0206 N + 1.7609); the **numerical output of the cells above was not executed by the author** — run them and confirm the printed differences are within ~0.3 dB.
* Everything else in this pack is derived from first principles; no standard-specific figures are used.